Наконец в этом ноутбуке мы будем делать регрессию для таргета SI.

In [1]:
!pip install catboost

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [4]:
df = pd.read_pickle('dataset_after_eda.pkl')
df = df.drop_duplicates()
df.shape

(958, 211)

In [5]:
y = np.log10(df['SI'])
X = df.drop(['IC50, mM', 'CC50, mM', 'SI', 'IC50_above_median', 'CC50_above_median', 'SI_above_median', 'SI_above_8'], axis=1)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape[0])
print(X_test.shape[0])

766
192


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("regressor", LinearRegression())]),
    "Ridge Regression": Pipeline([("scaler", StandardScaler()), ("regressor", Ridge())]),
    "Lasso Regression": Pipeline([("scaler", StandardScaler()), ("regressor", Lasso(alpha=0.1))]),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False) # verbose=0, чтобы не спамить логами
}

baseline_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_log = model.predict(X_test)

    mae_log = mean_absolute_error(y_test, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
    r2 = r2_score(y_test, y_pred_log)

    y_test_real = 10 ** y_test
    y_pred_real = 10 ** y_pred_log

    mae_real = mean_absolute_error(y_test_real, y_pred_real)

    baseline_results.append({
        "Model": name,
        "MAE (pIC50)": round(mae_log, 4),
        "RMSE (pIC50)": round(rmse_log, 4),
        "MAE (реальный, mM)": round(mae_real, 4),
        "R2 Score": round(r2, 4)
    })

df_baseline = pd.DataFrame(baseline_results).sort_values(by="R2 Score", ascending=False)
df_baseline

,Model,MAE (pIC50),RMSE (pIC50),"MAE (реальный, mM)",R2 Score
4,Gradient Boosting,0.5014,0.6356,17.9310,0.1919
3,Random Forest,0.4962,0.6478,17.0743,0.1606
5,CatBoost,0.4946,0.6484,17.0761,0.1589
1,Ridge Regression,0.5353,0.6837,19.1716,0.0648
2,Lasso Regression,0.5735,0.6953,18.7315,0.0328
0,Linear Regression,0.5703,0.7313,22.0084,-0.0698


Вспоминаем EDA (боксплоты), что и требовалось доказать.
Все модели показали низкий R2 на параметрах по умолчанию с максимумом на 0.19 у GB

In [8]:
grid = {
    'n_estimators': [300, 400],
    'max_depth': [12, 15],
    'min_samples_split': [3, 4],
    'min_samples_leaf': [2]
}

rf_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Random Forest")
rf_search.fit(X_train, y_train)

print(rf_search.best_params_)
best_rf_model = rf_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_rf))

Random Forest
Fitting 3 folds for each of 8 candidates, totalling 24 fits
{'max_depth': 12, 'min_samples_leaf': 2, 'min_samples_split': 3, 'n_estimators': 300}
R2: 0.19899527695545105


In [9]:
from sklearn.ensemble import GradientBoostingRegressor

gb_param_grid = {
    'n_estimators': [150, 300, 500],
    'learning_rate': [0.03, 0.05],
    'max_depth': [4, 5],
    'min_samples_leaf': [2, 3]
}

gb_grid_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gb_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Gradient Boosting")
gb_grid_search.fit(X_train, y_train)
print(gb_grid_search.best_params_)
best_gb_model = gb_grid_search.best_estimator_
y_pred_best_gb = best_gb_model.predict(X_test)

print("R2:", r2_score(y_test, y_pred_best_gb))

Gradient Boosting
Fitting 3 folds for each of 24 candidates, totalling 72 fits
{'learning_rate': 0.03, 'max_depth': 4, 'min_samples_leaf': 3, 'n_estimators': 150}
R2: 0.20862843936093967


In [10]:
from catboost import CatBoostRegressor

cat_param_grid = {
    'iterations': [300, 500],
    'learning_rate': [0.03, 0.05],
    'depth': [4, 6],
    'l2_leaf_reg': [3]
}

cat_grid_search = GridSearchCV(
    estimator=CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False),
    param_grid=cat_param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("CatBoost")
cat_grid_search.fit(X_train, y_train)
print(cat_grid_search.best_params_)
best_cat_model = cat_grid_search.best_estimator_
print(r2_score(y_test, best_cat_model.predict(X_test)))

CatBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
{'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}
0.24967717135812761


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

best_cat_si_model = cat_grid_search.best_estimator_
y_pred_log_cat = best_cat_si_model.predict(X_test)

mae_log = mean_absolute_error(y_test, y_pred_log_cat)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log_cat))
r2_si = r2_score(y_test, y_pred_log_cat)

y_test_real = 10 ** y_test
y_pred_real = 10 ** y_pred_log_cat

mae_real = mean_absolute_error(y_test_real, y_pred_real)

print("R2 ", r2_si)
print("MAE ",mae_log)
print("RMSE ",rmse_log)
print("MAE real",mae_real)

R2  0.24967717135812761
MAE  0.48497543953007954
RMSE  0.6124456001052431
MAE real 17.237534367491204


Random Forest:<br>
R2: 0.19893090825711257<br>

Gradient Boosting:<br>
R2: 0.20858355719436772<br>

CatBoost:<br>
R2 0.24967717135812761<br>
MAE 0.48497543953007954<br>
RMSE 0.6124456001052431<br>
MAE real 17.237534367491204

#Вывод

SI сложный признак, можно было ожидать от производной величины. Линейные модели показывают предикты на уровне 'наверное попал', сложный признак тем более будет плохо прогнозироваться.Зависимость между структурой молекулы и её селективностью это строго нелинейный характер. Лучшей моделью признан CatBoost с параметрами {'depth': 6, 'iterations': 300, 'l2_leaf_reg': 3, 'learning_rate': 0.03}. Он показал максимальный R2 = 0.2497 и обошел алгоритмы Gradient Boosting и Random Forest. Думаю можно сказать что модель по итогу способна улавливать паттерны и может использоваться для первичного скрининга ранжирования соединений.